In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import lora_transfer_pruning
from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map="cuda:3",
    # cache_dir="/glazkov-dev/.cache",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [4]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

In [5]:
bridge.model

LlamaModel(
  (embed_tokens): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (layers): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
          (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
          (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
          (hook_result): HookPoint(name='blocks.0.attn.hook_result')
          (hook_attn_in): HookPoint(name='blocks.0.attn.hook

In [6]:
bridge.named_modules()

<generator object Module.named_modules at 0x73a9c0202420>

In [7]:
for name, module in bridge.named_modules():
    print(name, type(module))

 <class 'transformer_lens.model_bridge.bridge.TransformerBridge'>
embed <class 'transformer_lens.model_bridge.generalized_components.embedding.EmbeddingBridge'>
embed.hook_in <class 'transformer_lens.hook_points.HookPoint'>
embed.hook_out <class 'transformer_lens.hook_points.HookPoint'>
embed._original_component <class 'torch.nn.modules.sparse.Embedding'>
rotary_emb <class 'transformer_lens.model_bridge.generalized_components.rotary_embedding.RotaryEmbeddingBridge'>
rotary_emb.hook_in <class 'transformer_lens.hook_points.HookPoint'>
rotary_emb.hook_out <class 'transformer_lens.hook_points.HookPoint'>
rotary_emb.hook_cos <class 'transformer_lens.hook_points.HookPoint'>
rotary_emb.hook_sin <class 'transformer_lens.hook_points.HookPoint'>
rotary_emb._original_component <class 'transformers.models.llama.modeling_llama.LlamaRotaryEmbedding'>
blocks <class 'torch.nn.modules.container.ModuleList'>
blocks.0 <class 'transformer_lens.model_bridge.generalized_components.block.BlockBridge'>
blocks

In [8]:
from transformer_lens.model_bridge.generalized_components.base import GeneralizedComponent


for name, module in bridge.named_modules():
    if (isinstance(module, GeneralizedComponent)):
        print(name, type(module))

embed <class 'transformer_lens.model_bridge.generalized_components.embedding.EmbeddingBridge'>
rotary_emb <class 'transformer_lens.model_bridge.generalized_components.rotary_embedding.RotaryEmbeddingBridge'>
blocks.0 <class 'transformer_lens.model_bridge.generalized_components.block.BlockBridge'>
blocks.0._original_component.self_attn <class 'transformer_lens.model_bridge.generalized_components.position_embeddings_attention.PositionEmbeddingsAttentionBridge'>
blocks.0._original_component.self_attn._original_component.q_proj <class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
blocks.0._original_component.self_attn._original_component.k_proj <class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
blocks.0._original_component.self_attn._original_component.v_proj <class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
blocks.0._original_component.self_attn._original_component.o_proj <class 'transformer_len

### NOTE: Inherinance realized a little bit strange, because generalized submodules embeded in `_original_components`.

In [9]:
bridge.adapter.component_mapping

{'embed': EmbeddingBridge(
   (hook_in): HookPoint(name='embed.hook_in')
   (hook_out): HookPoint(name='embed.hook_out')
   (_original_component): Embedding(128256, 4096)
 ),
 'rotary_emb': RotaryEmbeddingBridge(
   (hook_in): HookPoint(name='rotary_emb.hook_in')
   (hook_out): HookPoint(name='rotary_emb.hook_out')
   (hook_cos): HookPoint(name='rotary_emb.hook_cos')
   (hook_sin): HookPoint(name='rotary_emb.hook_sin')
   (_original_component): LlamaRotaryEmbedding()
 ),
 'blocks': BlockBridge(
   (hook_in): HookPoint(name='hook_in')
   (hook_out): HookPoint(name='hook_out')
   (hook_mlp_in): HookPoint(name='hook_mlp_in')
 ),
 'ln_final': RMSNormalizationBridge(
   (hook_in): HookPoint(name='ln_final.hook_in')
   (hook_out): HookPoint(name='ln_final.hook_out')
   (hook_normalized): HookPoint(name='ln_final.hook_normalized')
   (hook_scale): HookPoint(name='ln_final.hook_scale')
   (_original_component): LlamaRMSNorm((4096,), eps=1e-05)
 ),
 'unembed': UnembeddingBridge(
   (hook_in): H

from https://transformerlensorg.github.io/TransformerLens/content/model_structure.html

Shapes at a Glance

    Residual stream and hidden states: (batch, pos, d_model)

    Attention scores: (batch, n_heads, pos, pos)

    Attention patterns: (n_heads, pos, pos) - after batch dimension removal

    QKV projections: (batch, pos, n_heads, d_head)

    MLP pre-activation: (batch, pos, d_mlp)

    Embeddings: (batch, pos, d_model)

    Unembedding logits: (batch, pos, d_vocab)

    LayerNorm normalized / scale: (batch, pos, d_model)


From attention.py  
    def _setup_qkv_hook_reshaping(self) -> None:
        """Setup hook reshaping for Q/K/V/Z to match HookedTransformer shapes.

        Reshapes hooks from [batch, seq, d_model] to [batch, seq, n_heads, d_head] format.
        For models with Grouped Query Attention (GQA), K and V use n_kv_heads instead of n_heads.

        Sets up conversions for:
        - q.hook_out (aliased as hook_q)
        - k.hook_out (aliased as hook_k) - uses n_kv_heads if GQA
        - v.hook_out (aliased as hook_v) - uses n_kv_heads if GQA
        - o.hook_in (aliased as hook_z)
        """

It's usefull to see source code with many details for different architectures.

In [10]:
print(bridge.model.config._attn_implementation)

sdpa


In [11]:
from transformers import AttentionInterface, LlamaModel

from transformer_lens.components.embed import Embed #old, deprecated
from transformer_lens.conversion_utils.conversion_steps.base_tensor_conversion import BaseTensorConversion
from transformer_lens.hook_points import HookPoint
from transformer_lens.model_bridge.generalized_components.attention import AttentionBridge
from transformer_lens.model_bridge.generalized_components.block import BlockBridge
from transformer_lens.model_bridge.generalized_components.embedding import EmbeddingBridge
from transformer_lens.model_bridge.generalized_components.linear import LinearBridge
from transformer_lens.model_bridge.generalized_components.position_embeddings_attention import PositionEmbeddingsAttentionBridge
from transformer_lens.model_bridge.generalized_components.rotary_embedding import RotaryEmbeddingBridge

LlamaModel,
AttentionInterface

EmbeddingBridge, #dont want to prune now (defenetly not rows)
RotaryEmbeddingBridge, #doesnt contain any weights
BlockBridge,
PositionEmbeddingsAttentionBridge,
AttentionBridge, #uses qkv hook reshaping (hook conversion) to 4D tensors
HookPoint, BaseTensorConversion, #contains hook_conversion that changes shape
#HookPoint(nn.Module) can contains many fwd and bkw pytorch hooks via .register_forward_hook on that "module"
#HookPoint used in Bridged layers (ussualy as layer.hook_in, layer.hook_out)
#layer.hook_out.hook_conversion or hook_out.enable_reshape()
#for example hook_in = HookPoint; x = hook_in(x); x = original_component(x); hook_out = HookPoint; x = hook_out(x); return x;


LinearBridge

transformer_lens.model_bridge.generalized_components.linear.LinearBridge

In [12]:
print(bridge.blocks[0].attn.config.use_split_qkv_input)
print(bridge.blocks[0].attn.config.use_attn_in)

False
False


In [13]:
bridge.blocks[0].attn.q

LinearBridge(4096 -> 4096, bias=False, original_component=Linear)

### Inside TransformerLens attention:
https://transformerlensorg.github.io/TransformerLens/generated/code/transformer_lens.config.hooked_transformer_config.html

use_hook_mlp_in (bool) – whether to use a hook to get the input to the MLP layer. Defaults to `false` to save memory.

use_attn_in (bool) – whether to explicitly calculate the input of each attention head separately, with a hook. Defaults to `false` to save memory

(before attn.o)  
use_attn_result (bool) – whether to explicitly calculate the amount each head adds to the residual stream (with a hook) and THEN add it up, vs just calculating the sum. This can be very memory intensive for large models, so defaults to `False`

In [14]:
import importlib
importlib.reload(lora_transfer_pruning.core.pruning_instrumentor)

<module 'lora_transfer_pruning.core.pruning_instrumentor' from '/glazkov-dev/LoRa-Transfer-Pruning/lora_transfer_pruning/core/pruning_instrumentor.py'>

## Оценка качества до и после pruning

Используем validation split WikiText-2 и сравниваем средний next-token loss и perplexity на одних и тех же блоках токенов.

In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [16]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

available_blocks = validation_tokens.numel() // CONTEXT_LENGTH
num_blocks = min(NUM_EVAL_BLOCKS, available_blocks)
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [17]:
def evaluate_language_model(evaluated_bridge, token_blocks, batch_size=1):
    evaluated_bridge.eval()
    model_device = next(evaluated_bridge.parameters()).device
    losses = []

    with torch.inference_mode():
        for start in range(0, len(token_blocks), batch_size):
            batch = token_blocks[start : start + batch_size].to(model_device)
            loss = evaluated_bridge.run_with_hooks(batch, return_type="loss")
            losses.append(loss.detach().float().cpu())

    mean_loss = torch.stack(losses).mean()
    return {
        "loss": mean_loss.item(),
        "perplexity": mean_loss.exp().item(),
    }

bridge.reset_hooks()
baseline_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
baseline_metrics

{'loss': 2.26513671875, 'perplexity': 9.632441520690918}

In [18]:
modules_to_prune = []

for module in bridge.modules():
    if (type(module) == transformer_lens.model_bridge.generalized_components.linear.LinearBridge):
        if (module.name not in ["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"]):
            modules_to_prune.append(module)

print(set([i.name for i in modules_to_prune]))
print("contains BIAS", [i._original_component.bias for i in modules_to_prune])
print(len(modules_to_prune))
#embed - EmbeddingBridge
#ln_head - RMSNormalizationBridge
#unembed - UnembeddingBridge

{'gate_proj', 'up_proj'}
contains BIAS [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
64


In [55]:
bridge.reset_hooks() #doesnt remove them
# bridge.blocks[0].remove_all_hook_fns(including_permanent=True) 
#.remove_all_hook_fns(including_permanent=True) work only for HookPoint classes

In [56]:

for module in modules_to_prune:
    PruningInstrumentor.prepare_linear_for_pruning_fraction(module, 0, 0.1)

In [57]:
bridge.blocks[0].attn.q.hook_in.fwd_hooks

[]

In [58]:
for module in bridge.modules():
    if (type(module) == transformer_lens.model_bridge.generalized_components.linear.LinearBridge):
        print(module.weight.shape)

torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size([14336, 4096])
torch.Size([14336, 4096])
torch.Size([4096, 14336])
torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size([14336, 4096])
torch.Size([14336, 4096])
torch.Size([4096, 14336])
torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size([14336, 4096])
torch.Size([14336, 4096])
torch.Size([4096, 14336])
torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size([14336, 4096])
torch.Size([14336, 4096])
torch.Size([4096, 14336])
torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size([14336, 4096])
torch.Size([14336, 4096])
torch.Size([4096, 14336])
torch.Size([4096, 4096])
torch.Size([1024, 4096])
torch.Size([1024, 4096])
torch.Size([4096, 4096])
torch.Size

In [59]:
print(bridge.model.config.num_key_value_heads)
print(bridge.model.config.num_attention_heads)
print(bridge.model.config.head_dim)
print(bridge.model.config._attn_implementation)

8
32
128
sdpa


In [60]:
pruned_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
comparison = {
    "without_pruning": baseline_metrics,
    "with_pruning": pruned_metrics,
    "loss_delta": pruned_metrics["loss"] - baseline_metrics["loss"],
    "perplexity_ratio": (
        pruned_metrics["perplexity"] / baseline_metrics["perplexity"]
    ),
}
comparison

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 3.205078125, 'perplexity': 24.657426834106445},
 'loss_delta': 0.93994140625,
 'perplexity_ratio': 2.559831459255806}

## Without rescaling after prune (better?)

In [61]:
modules_to_prune = []

for module in bridge.modules():
    if (type(module) == transformer_lens.model_bridge.generalized_components.linear.LinearBridge):
        if (module.name not in ["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"]):
            modules_to_prune.append(module)

print(set([i.name for i in modules_to_prune]))
print([i._original_component.bias for i in modules_to_prune])
print(len(modules_to_prune))


bridge.reset_hooks()

#make without rescaling
PruningInstrumentor._rescale_activation_after_ablation = lambda new_activation, old_activation, dim=-1: new_activation

for module in modules_to_prune:
    PruningInstrumentor.prepare_linear_for_pruning_fraction(module, 0, 0.1)
        
        
pruned_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
comparison = {
    "without_pruning": baseline_metrics,
    "with_pruning": pruned_metrics,
    "loss_delta": pruned_metrics["loss"] - baseline_metrics["loss"],
    "perplexity_ratio": (
        pruned_metrics["perplexity"] / baseline_metrics["perplexity"]
    ),
}
comparison

{'up_proj', 'gate_proj'}
[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
64


{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 3.0634765625, 'perplexity': 21.401832580566406},
 'loss_delta': 0.79833984375,
 'perplexity_ratio': 2.2218492097350713}

## Only without sensetive modules v_proj, down_proj

In [69]:
def prune_model(cols_fraction=0, rows_fraction=0.1, dont_prune=["v_proj", "down_proj"]):
    from lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor

    modules_to_prune = []

    for module in bridge.modules():
        if (type(module) == transformer_lens.model_bridge.generalized_components.linear.LinearBridge):
            if (module.name not in dont_prune):
                modules_to_prune.append(module)

    # print(set([i.name for i in modules_to_prune]))
    # print([i._original_component.bias for i in modules_to_prune])
    # print(len(modules_to_prune))


    bridge.reset_hooks()


    for module in modules_to_prune:
        PruningInstrumentor.prepare_linear_for_pruning_fraction(module, cols_fraction, rows_fraction)
            
            
    pruned_metrics = evaluate_language_model(
        bridge,
        evaluation_blocks,
        batch_size=EVAL_BATCH_SIZE,
    )
    comparison = {
        "without_pruning": baseline_metrics,
        "with_pruning": pruned_metrics,
        "loss_delta": pruned_metrics["loss"] - baseline_metrics["loss"],
        "perplexity_ratio": (
            pruned_metrics["perplexity"] / baseline_metrics["perplexity"]
        ),
    }
    return comparison

In [70]:
prune_model(0, 0.1)

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 7.2802734375, 'perplexity': 1451.3848876953125},
 'loss_delta': 5.01513671875,
 'perplexity_ratio': 150.67674011595838}

## Without attention and more pruning_fraction

In [71]:
prune_model(0, 0.1, dont_prune=["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 3.201171875, 'perplexity': 24.561296463012695},
 'loss_delta': 0.93603515625,
 'perplexity_ratio': 2.549851604108255}

In [72]:
prune_model(0.1, 0.1, dont_prune=["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 4.0498046875, 'perplexity': 57.38624572753906},
 'loss_delta': 1.78466796875,
 'perplexity_ratio': 5.957601258649826}

In [73]:
prune_model(0.2, 0.2, dont_prune=["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 8.123046875, 'perplexity': 3371.27685546875},
 'loss_delta': 5.85791015625,
 'perplexity_ratio': 349.9919359206174}

In [74]:
prune_model(0.1, 0, dont_prune=["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 2.517333984375, 'perplexity': 12.395505905151367},
 'loss_delta': 0.252197265625,
 'perplexity_ratio': 1.2868498478319605}

In [75]:
prune_model(0.2, 0, dont_prune=["v_proj", "down_proj", "q_proj", "o_proj", "k_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 2.8759765625, 'perplexity': 17.74274253845215},
 'loss_delta': 0.61083984375,
 'perplexity_ratio': 1.841977706310486}

In [76]:
prune_model(0.2, 0, dont_prune=["v_proj", "down_proj"])

{'without_pruning': {'loss': 2.26513671875, 'perplexity': 9.632441520690918},
 'with_pruning': {'loss': 4.6162109375, 'perplexity': 101.11019134521484},
 'loss_delta': 2.35107421875,
 'perplexity_ratio': 10.49683936601386}

TODO: check correctness of pruning, scaling, bias, rows/columns and so on.

Or check with microsoft nni/other libs, that quality degradation on weight pruning equal degradation on activation pruning.